Format csvs into JSON.

In [4]:
import pandas as pd

# Convert CSV files into dataframes
products = pd.read_csv('../dataset/products.csv')
aisles = pd.read_csv('../dataset/aisles.csv')
departments = pd.read_csv('../dataset/departments.csv')
sampled_products = pd.read_csv('../results/sampled-products.csv')

products_full = (products
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)

filtered_products = products_full[products_full['product_id'].isin(sampled_products['product_id'])]

print(filtered_products.head())

      product_id                       product_name  aisle_id  department_id  \
505          506                   Arrowroot Powder         6              2   
2381        2382                     Organic Eggnog        84             16   
5334        5335  Organic Fresh Pressed Apple Juice        98              7   
6183        6184                        Clementines        32              4   
6454        6455               Organic Magic Muesli        68             10   

                             aisle  department  
505                          other       other  
2381                          milk  dairy eggs  
5334                 juice nectars   beverages  
6183              packaged produce     produce  
6454  bulk grains rice dried goods        bulk  


In [2]:
import pandas as pd
import json

df = filtered_products.copy()

# -------------------------------
# 1️⃣ Departments → Aisles
# -------------------------------
departments = []
for (dept_id, dept_name), dept_group in df.groupby(['department_id', 'department']):
    aisles = [
        {
            'aisle_id': int(a_id),
            'aisle': a_name
        }
        for (a_id, a_name) in dept_group[['aisle_id', 'aisle']].drop_duplicates().to_records(index=False)
    ]
    departments.append({
        'department_id': int(dept_id),
        'department': dept_name,
        'aisles': aisles
    })

# -------------------------------
# 2️⃣ Aisles → Products
# -------------------------------
aisles = []
for (aisle_id, aisle_name), aisle_group in df.groupby(['aisle_id', 'aisle']):
    products = [
        {
            'product_id': int(pid),
            'product_name': pname
        }
        for pid, pname in aisle_group[['product_id', 'product_name']].to_records(index=False)
    ]
    aisles.append({
        'aisle_id': int(aisle_id),
        'aisle': aisle_name,
        'products': products
    })

# -------------------------------
# 💾 Save both to JSON files
# -------------------------------
with open('../json/sample-departments.json', 'w') as f:
    json.dump(departments, f, indent=2)

#with open('../json/sample-aisles.json', 'w') as f:
#    json.dump(aisles, f, indent=2)

print("✅ JSON files created: departments.json and aisles.json")

✅ JSON files created: departments.json and aisles.json


In [6]:
import pandas as pd
import json

df = filtered_products.copy()

# -------------------------------
# Departments → Products
# -------------------------------
departments = []
for (dept_id, dept_name), dept_group in df.groupby(['department_id', 'department']):
    products = [
        {
            'product_id': int(pid),
            'product_name': pname
        }
        for pid, pname in dept_group[['product_id', 'product_name']].to_records(index=False)
    ]
    departments.append({
        'department_id': int(dept_id),
        'department': dept_name,
        'products': products
    })


with open('../json/sample-departments.json', 'w') as f:
    json.dump(departments, f, indent=2)

print("✅ JSON files created: departments.json and aisles.json")

✅ JSON files created: departments.json and aisles.json


In [7]:
import pandas as pd
import json

products = pd.read_csv('../dataset/products.csv')
# Convert product_id to int to avoid JSON serialization issues
products['product_id'] = products['product_id'].astype(int)

# Create dictionary: {product_id: product_name}
product_lookup = dict(zip(products['product_id'], products['product_name']))

# Save as JSON
with open('../json/product-lookup.json', 'w') as f:
    json.dump(product_lookup, f, indent=2)

print("✅ Created product_lookup.json")


✅ Created product_lookup.json


In [11]:
import pandas as pd
import json
sampled_products = pd.read_csv('../data/results/sampled-products.csv')


all_subs = pd.read_csv('../data/results/obj1/substitutes-4-full.csv')

df = all_subs[all_subs['identified_substitute'] == True]

# Make sure column names are correct
required_cols = ['product_id', 'substitute_id', 'score', 'rank']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure proper Python-native types (avoid numpy int64/float64)
df = df.astype({
    'product_id': 'int64',
    'substitute_id': 'int64',
    'score': 'float64',
    'rank': 'int64'
}, errors='ignore')

# Group and build the JSON
subs_lookup = (
    df.groupby('product_id', group_keys=False)
      .apply(lambda g: [
          {
              "product_id": int(row['substitute_id']),
              "score": float(row['score']),
              "rank": int(row['rank'])
          }
          for _, row in g.iterrows()
      ])
      .to_dict()
)

# Convert keys to strings for JSON
subs_lookup = {str(k): v for k, v in subs_lookup.items()}

# Save to JSON file
with open('../json/product_true_substitutes.json', 'w') as f:
    json.dump(subs_lookup, f, indent=2)

print("✅ Created product_substitutes.json successfully.")



C:\Users\jenle\AppData\Local\Temp\ipykernel_25336\4082690263.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: [


✅ Created product_substitutes.json successfully.


In [13]:
import pandas as pd
import json

sampled_products = pd.read_csv('../results/sampled-products.csv')


df = pd.read_csv('../results/complements.csv')

# Make sure column names are correct
required_cols = ['product_id', 'complement_id', 'CII', 'lift']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure proper Python-native types (avoid numpy int64/float64)
df = df.astype({
    'product_id': 'int64',
    'complement_id': 'int64',
    'CII': 'float64',
    'lift': 'float64'
}, errors='ignore')

# Group and build the JSON
comps_lookup = (
    df.groupby('product_id', group_keys=False)
      .apply(lambda g: [
          {
              "product_id": int(row['complement_id']),
              "CII": float(row['CII']),
              "lift": int(row['lift'])
          }
          for _, row in g.iterrows()
      ])
      .to_dict()
)

# Convert keys to strings for JSON
comps_lookup = {str(k): v for k, v in comps_lookup.items()}

# Save to JSON file
with open('../json/complements.json', 'w') as f:
    json.dump(comps_lookup, f, indent=2)

print("✅ Created complements.json successfully.")



✅ Created complements.json successfully.


C:\Users\jenle\AppData\Local\Temp\ipykernel_18872\3813934324.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: [


In [19]:
import pandas as pd
import json

def build_product_graph(products_df, transfer_df, comps_df, total_impact_df):
    """
    Build hierarchical JSON object:

    {
        product_id: {
            "name": str,
            "values": [...],            # (top-level product values if any)
            "subs": [                   # substitute objects
                {
                    "id": str,
                    "name": str,
                    "values": [...],    # [score, transferability_pct]
                    "comps": [ ... ]    # comps for this sub (from subs_comps_df)
                }
            ],
            "comps": [                  # complements for the product
                {
                    "id": str,
                    "name": str,
                    "values": [...]     # [impact_pct_j, impact_index_j]
                }
            ]
        },
        ...
    }
    """

    # Make product lookup for fast name resolution
    product_name_lookup = (
        products_df
        .set_index("product_id")["product_name"]
        .to_dict()
    )

    dept_lookup = (
        products_df
        .set_index("product_id")["department"]
        .to_dict()
    )

    aisle_lookup = (
        products_df
        .set_index("product_id")["aisle"]
        .to_dict()
    )

    # Initialize object
    output = {}

    # Prepare groupings for quick lookup
    transfer_groups = transfer_df.groupby("product_id")
    comps_groups = comps_df.groupby("product_id")

    focus_products = transfer_df['product_id'].unique()

    # Iterate over all products
    for pid in focus_products:
        pid_str = str(pid)
        pname = product_name_lookup[pid]

        # Top-level structure
        output[pid_str] = {
            "name": pname,
            "dept": dept_lookup[pid],
            "aisle": aisle_lookup[pid],
            "values": [],  # You can fill this later if needed
            "subs": [],
            "comps": []
        }
        total_transfer = 0
        # -----------------------------
        # 1. SUBSTITUTES
        # -----------------------------
        if pid in transfer_groups.groups:
            sub_rows = transfer_groups.get_group(pid)
            for _, row in sub_rows.iterrows():
                sid = str(row["substitute_id"])
                sub_obj = {
                    "id": sid,
                    "name": product_name_lookup.get(row["substitute_id"], ""),
                    "values": [
                        round(float(row["score"]), 3),
                        round(float(row["transferability_pct"]),3)
                    ],
                    "comps": []
                }

                total_transfer += row["transferability_pct"]

                # ---- SUB'S COMPLEMENTS ----
                # These come from subs_comps_df where product_id = pid AND complement refers to a sub's complement
                # subs_comps_df is grouped by product_id for lookup
                # But we must filter rows where product_j == substitute_id
                sub_id = row["substitute_id"]
                if sub_id in comps_groups.groups:
                    sub_comp_rows = comps_groups.get_group(sub_id)

                    for _, crow in sub_comp_rows.iterrows():
                        cid = str(crow["complement_id"])  # complement id associated to subs
                        sub_obj["comps"].append({
                            "id": cid,
                            "name": product_name_lookup.get(crow["complement_id"], ""),
                            "dept": dept_lookup.get(crow["complement_id"], ""),
                            "aisle": aisle_lookup.get(crow["complement_id"], ""),
                            "values": [
                                round(float(crow["hybrid_score"]),3),
                                round(float(crow["impact_index"])*100, 1)
                            ]
                        })

                # Add the sub object
                output[pid_str]["subs"].append(sub_obj)

        output[pid_str]["values"].append(round(total_transfer, 3))

        # -----------------------------
        # 2. PRODUCT-LEVEL COMPLEMENTS
        # -----------------------------
        if pid in comps_groups.groups:
            comp_rows = comps_groups.get_group(pid)
            for _, row in comp_rows.iterrows():
                cid = str(row["complement_id"])
                comp_obj = {
                    "id": cid,
                    "name": product_name_lookup.get(row["complement_id"], ""),
                    "dept": dept_lookup.get(row["complement_id"], ""),
                    "aisle": aisle_lookup.get(row["complement_id"], ""),
                    "values": [
                        round(float(row["hybrid_score"]),3),
                        round(float(row["impact_index"])*100, 1)
                    ]
                }
                output[pid_str]["comps"].append(comp_obj)

        total_impact = round(total_impact_df.loc[total_impact_df.product_id == pid, 'total_impact_norm'].iloc[0]*100, 1)

        output[pid_str]["values"].append(total_impact)

    with open('../json/focus-products.json', 'w') as f:
        json.dump(output, f, indent=2)

products_df = pd.read_csv('../dataset/products.csv')
transfer_df = pd.read_csv('../results/substitutes-transfer.csv')
comps_df = pd.read_csv('../results/complements-impact.csv')
aisles = pd.read_csv('../dataset/aisles.csv')
departments = pd.read_csv('../dataset/departments.csv')
total_impact_df = pd.read_csv('../results/complement-total-impact.csv')

products_full = (products_df
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)

build_product_graph(products_full, transfer_df, comps_df, total_impact_df)